# Visual Attendance System (v3, corrected)
ArcFace (InsightFace) recognition, guided enrollment, stable-frame marking, SQLite log, Excel export.
**Run locally in Jupyter/VS Code** (OpenCV windows do not work in Colab). Uses the same `attendance.db` as `app.py`, so the notebook and the web dashboard share students and logs.

Order: cell 1 (install, once) -> 2 -> 3 -> 4 -> 5 -> 6 -> 7, then use cell 8.

In [1]:
%pip install -q insightface onnxruntime opencv-python numpy pandas openpyxl

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, sqlite3
from collections import defaultdict
from datetime import datetime
import cv2, numpy as np, pandas as pd

DB = "attendance.db"
STUDENTS_DIR = "students"     # enrollment photos are saved here for auditing
CAMERA_INDEX = 0              # try 1 if the wrong camera opens

# --- Recognition tuning ---
THRESH = 0.45        # cosine similarity needed to accept a match (0.50 = stricter)
MARGIN = 0.08        # best match must beat the runner-up by this much
MIN_DET = 0.60       # minimum detector confidence
MIN_FACE = 80        # minimum face width in pixels (on the 640px-wide frame)
BLUR_MIN = 60        # reject blurry enrollment photos
NUM_PHOTOS = 8

# --- Attendance tuning ---
STABLE = 3           # consecutive recognized frames before marking
LATE_AFTER = "08:30" # HH:MM, later than this is marked Late
PROCESS_EVERY = 2    # run the model on every Nth frame (speed)

In [3]:
from insightface.app import FaceAnalysis

# First run downloads the buffalo_l model (~300 MB). For GPU use CUDAExecutionProvider + pip install onnxruntime-gpu
fa = FaceAnalysis(name="buffalo_l", providers=["CPUExecutionProvider"])
fa.prepare(ctx_id=-1, det_size=(640, 640))

db = sqlite3.connect(DB)
db.executescript(
    "CREATE TABLE IF NOT EXISTS students(roll TEXT PRIMARY KEY, name TEXT);"
    "CREATE TABLE IF NOT EXISTS emb(roll TEXT, v BLOB);"
    "CREATE TABLE IF NOT EXISTS log(day TEXT, t TEXT, roll TEXT, status TEXT, conf REAL, PRIMARY KEY(day, roll));"
)

G_V, G_R = np.zeros((0, 512), np.float32), []

def load_gallery():
    global G_V, G_R
    rows = db.execute("SELECT roll, v FROM emb").fetchall()
    G_R = [r for r, _ in rows]
    G_V = np.array([np.frombuffer(v, np.float32) for _, v in rows]).reshape(-1, 512)
    print(f"Gallery: {len(set(G_R))} people, {len(G_R)} embeddings")

def faces_in(img):
    # detect on a 640px-wide copy for speed, then scale boxes back
    h, w = img.shape[:2]
    s = 640 / w
    small = cv2.resize(img, (640, int(h * s))) if w != 640 else img
    out = [f for f in fa.get(small) if f.det_score >= MIN_DET and (f.bbox[2] - f.bbox[0]) >= MIN_FACE]
    return out, s

def identify(emb):
    # best person by max cosine similarity, and must beat the runner-up by MARGIN
    if not len(G_R):
        return None, 0.0
    per = {}
    for r, s in zip(G_R, G_V @ emb):
        per[r] = max(per.get(r, -1.0), float(s))
    ranked = sorted(per.items(), key=lambda x: -x[1])
    best_r, best_s = ranked[0]
    second = ranked[1][1] if len(ranked) > 1 else -1.0
    if best_s >= THRESH and best_s - second >= MARGIN:
        return best_r, best_s
    return None, best_s

load_gallery()

Gallery: 0 people, 0 embeddings


In [4]:
def _guidance(bbox, shape):
    l, t, r, b = bbox
    h, w = shape[:2]
    msgs = []
    dx, dy = (l + r) / 2 - w / 2, (t + b) / 2 - h / 2
    if dx > w * 0.12: msgs.append("Move LEFT")
    elif dx < -w * 0.12: msgs.append("Move RIGHT")
    if dy > h * 0.12: msgs.append("Move UP")
    elif dy < -h * 0.12: msgs.append("Move DOWN")
    ratio = (r - l) / w
    if ratio < 0.22: msgs.append("Move CLOSER")
    elif ratio > 0.55: msgs.append("Move BACK")
    return (" | ".join(msgs), False) if msgs else ("Ready - press SPACE", True)


def enroll_student(roll_no, name, num_photos=NUM_PHOTOS, overwrite=False):
    # SPACE = capture (only when centered, sharp, and exactly one face). Q = stop.
    roll = str(roll_no).strip()
    name = " ".join(str(name).split())
    if not roll or not name:
        print("Roll number and name are both required."); return
    row = db.execute("SELECT name FROM students WHERE roll=?", (roll,)).fetchone()
    if row and not overwrite:
        print(f"Roll {roll} already belongs to '{row[0]}'. Use a new roll number or overwrite=True."); return
    if row:
        db.execute("DELETE FROM emb WHERE roll=?", (roll,))

    folder = os.path.join(STUDENTS_DIR, f"{roll}_{name.replace(' ', '_').lower()}")
    os.makedirs(folder, exist_ok=True)
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("Could not open the camera."); return

    good, embs = 0, []
    print(f"Enrolling {name} (roll {roll}). Turn your head slightly between photos.")
    while good < num_photos:
        ok, frame = cap.read()
        if not ok: break
        faces, s = faces_in(frame)
        view, ready, text = frame.copy(), False, "No face detected"
        if len(faces) > 1:
            text = "Only one person in frame"
        elif len(faces) == 1:
            box = (faces[0].bbox / s).astype(int)
            text, ready = _guidance(box, frame.shape)
            cv2.rectangle(view, (box[0], box[1]), (box[2], box[3]), (0, 255, 0) if ready else (0, 200, 255), 2)
        cv2.putText(view, f"{name}: {good}/{num_photos}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(view, text, (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0) if ready else (0, 200, 255), 2)
        cv2.imshow("Enroll", view)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q'): break
        if key == ord(' ') and ready:
            if cv2.Laplacian(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var() < BLUR_MIN:
                print("  Too blurry, hold still."); continue
            emb = faces[0].normed_embedding.astype(np.float32)
            other, sim = identify(emb)
            if other and other != roll:
                print(f"  This face already matches roll {other} (similarity {sim:.2f}). Stopping."); break
            embs.append(emb); good += 1
            cv2.imwrite(os.path.join(folder, f"{good}.jpg"), frame)
            print(f"  Captured {good}/{num_photos}")
    cap.release(); cv2.destroyAllWindows()

    if len(embs) >= 3:
        db.executemany("INSERT INTO emb VALUES(?,?)", [(roll, e.tobytes()) for e in embs])
        db.execute("INSERT OR REPLACE INTO students VALUES(?,?)", (roll, name))
        db.commit(); load_gallery()
        print(f"Enrolled {name} with {len(embs)} photos.")
    else:
        db.rollback()
        print("Not enough good photos (need at least 3). Nothing saved.")


def delete_student(roll_no):
    roll = str(roll_no).strip()
    for t in ("emb", "students"):
        db.execute(f"DELETE FROM {t} WHERE roll=?", (roll,))
    db.commit(); load_gallery()

In [5]:
def mark_attendance(roll, conf, now):
    status = "Late" if now.strftime("%H:%M") > LATE_AFTER else "Present"
    cur = db.execute("INSERT OR IGNORE INTO log VALUES(?,?,?,?,?)",
                     (now.strftime("%Y-%m-%d"), now.strftime("%H:%M:%S"), roll, status, float(conf)))
    db.commit()
    return status if cur.rowcount == 1 else None   # None = already marked today


def daily_report(day=None):
    # every enrolled student for the day: Present / Late / Absent
    day = day or datetime.now().strftime("%Y-%m-%d")
    st = pd.read_sql("SELECT roll AS RollNo, name AS Name FROM students", db)
    lg = pd.read_sql("SELECT roll AS RollNo, t AS Time, status AS Status, conf AS Match FROM log WHERE day=?", db, params=(day,))
    df = st.merge(lg, on="RollNo", how="left")
    df["Status"] = df["Status"].fillna("Absent")
    df.insert(0, "Date", day)
    return df.sort_values(["Status", "Time"]).reset_index(drop=True)


def export_report(day=None):
    df = daily_report(day)
    path = f"attendance_{df['Date'].iloc[0] if len(df) else datetime.now():%Y-%m-%d}.xlsx"
    df.to_excel(path, index=False)
    print("Saved", path)
    return df

In [6]:
def run_attendance():
    load_gallery()
    if not len(G_R):
        print("No students enrolled yet. Run enroll_student(...) first."); return
    cap = cv2.VideoCapture(CAMERA_INDEX)
    if not cap.isOpened():
        print("Could not open the camera."); return

    streak = defaultdict(int)
    last, n = [], 0
    print("Running. Press Q in the video window to stop.")
    while True:
        ok, frame = cap.read()
        if not ok: break
        n += 1
        if n % PROCESS_EVERY == 0:
            faces, s = faces_in(frame)
            last, seen = [], set()
            for f in faces:
                roll, sim = identify(f.normed_embedding)
                box = (f.bbox / s).astype(int)
                if roll:
                    seen.add(roll)
                    streak[roll] += 1
                    name = db.execute("SELECT name FROM students WHERE roll=?", (roll,)).fetchone()[0]
                    label, color = f"{name} {sim:.0%}", (0, 255, 0)
                    if streak[roll] >= STABLE:
                        status = mark_attendance(roll, sim, datetime.now())
                        if status: print(f"[{datetime.now():%H:%M:%S}] {name}: {status}")
                else:
                    label, color = "Unknown", (0, 0, 255)
                last.append((box, label, color))
            for r in list(streak):
                if r not in seen: streak[r] = 0
        for box, label, color in last:
            cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3]), color, 2)
            cv2.putText(frame, label, (box[0], box[1] - 8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        cv2.imshow("Attendance", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break
    cap.release(); cv2.destroyAllWindows()
    print("Stopped. Use export_report() for the Excel sheet.")

In [8]:
# --- Usage (uncomment one line at a time) ---
# 1) Enroll each student ONCE. Every student needs a UNIQUE roll number.
enroll_student(1, "Cherinet Asrat")
# enroll_student(2, "Sara Ayele")
# enroll_student(3, "Jiksa Sagni")

# 2) Run live attendance (Q to stop)
# run_attendance()

# 3) See today's Present / Late / Absent and save it to Excel
# daily_report()
# export_report()

# Remove someone:  delete_student(2)

Enrolling Cherinet Asrat (roll 1). Turn your head slightly between photos.


C:\Users\hp\Desktop\project\visual attendance_ass\venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


  Captured 1/8
  Captured 2/8
  Captured 3/8
  Captured 4/8
  Captured 5/8
  Captured 6/8
  Captured 7/8
  Captured 8/8
Gallery: 1 people, 8 embeddings
Enrolled Cherinet Asrat with 8 photos.


In [9]:
run_attendance()

Gallery: 1 people, 8 embeddings
Running. Press Q in the video window to stop.
[08:08:40] Cherinet Asrat: Present
Stopped. Use export_report() for the Excel sheet.
